# Week 2 — 움직임 감지 촬영 기능
**환경**: Python 3.10.11 / Windows / VSCode Jupyter

**동작**
- 이전 프레임과 현재 프레임의 픽셀 차이를 계산
- 차이가 임계값(MOTION_THRESHOLD)을 넘으면 이벤트 사진 저장
- 정기 촬영과 같은 순간엔 중복 저장 방지
- 저장 파일명: `{timestamp}_motion_color.png` / `{timestamp}_motion_depth.png`

**실행 순서**: 셀을 위에서부터 순서대로 실행하세요 (Shift+Enter)

## 0. 패키지 설치
처음 한 번만 실행하면 됩니다

In [ ]:
%pip install opencv-python numpy matplotlib

## 1. 라이브러리 로드 & Kinect 연결 확인

In [ ]:
import cv2
import numpy as np
import time
import sys
from datetime import datetime
from pathlib import Path

sys.path.insert(0, "..")     # backend/ → utils 패키지 접근용
sys.path.insert(0, "../..")  # 프로젝트 루트 → config.py 접근용

from utils.db import upload_image

# Kinect 연결 시도
try:
    import pykinect_azure as pykinect
    KINECT_AVAILABLE = True
    print("[OK] Kinect 사용 가능")
except ImportError:
    KINECT_AVAILABLE = False
    print("[웹캠 모드] pykinect_azure 없음 → 웹캠으로 대체 실행")

print(f"OpenCV 버전: {cv2.__version__}")

## 2. 설정값

In [ ]:
# =============================================
#  설정값 — 필요 시 여기서 수정
# =============================================

INTERVAL_SEC     = 60    # 정기 촬영 간격 (초) — 테스트: 10 / 실제: 60
MOTION_THRESHOLD = 20    # 움직임 감지 민감도
                         # 낮을수록 민감 (어두운 환경에선 40~50 권장)
SAVE_BASE_DIR    = Path("./sleep_frames")
SHOW_PREVIEW     = True
MOTION_COOLDOWN_SEC = 10  # 움직임 감지 후 이 시간 동안 추가 촬영 안 함


# 저장 폴더 생성
today    = datetime.now().strftime("%Y%m%d")
SAVE_DIR = SAVE_BASE_DIR / today
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f"정기 촬영 간격 : {INTERVAL_SEC}초")
print(f"움직임 임계값  : {MOTION_THRESHOLD}")
print(f"저장 폴더      : {SAVE_DIR.resolve()}")

## 3. 공통 함수 정의

In [ ]:
def save_images(color_image, depth_image, label: str) -> str:
    """
    label: 'regular' 또는 'motion'
    반환: Storage 공개 URL (업로드 실패 시 로컬 경로)
    """
    timestamp  = int(time.time())
    color_path = SAVE_DIR / f"{timestamp}_{label}_color.png"
    depth_path = SAVE_DIR / f"{timestamp}_{label}_depth.png"
    cv2.imwrite(str(color_path), color_image)
    cv2.imwrite(str(depth_path), depth_image)
    return upload_image(str(color_path))


def depth_to_colormap(depth_image) -> np.ndarray:
    """뎁스 이미지를 컬러맵으로 변환"""
    depth_norm = cv2.normalize(
        depth_image, None, 0, 255, cv2.NORM_MINMAX
    ).astype(np.uint8)
    return cv2.applyColorMap(depth_norm, cv2.COLORMAP_JET)


def detect_motion(prev_gray, curr_gray) -> tuple[bool, float]:
    """
    이전/현재 프레임을 비교해서 움직임 여부와 점수를 반환
    반환: (움직임 감지 여부, 차이 점수)
    """
    diff  = cv2.absdiff(prev_gray, curr_gray)
    score = float(np.mean(diff))
    return score > MOTION_THRESHOLD, score


def draw_overlay(image, next_in: float,
                 regular_count: int, motion_count: int,
                 score: float) -> np.ndarray:
    """미리보기에 촬영 정보 오버레이"""
    overlay = image.copy()
    h, w    = overlay.shape[:2]

    cv2.rectangle(overlay, (0, 0), (w, 75), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.45, image, 0.55, 0, overlay)

    cv2.putText(overlay,
        f"다음 정기 촬영까지: {int(next_in)}초  |  정기: {regular_count}장  움직임: {motion_count}장",
        (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2)

    bar_w    = int((score / 100) * (w - 20))
    bar_w    = min(bar_w, w - 20)
    bar_col  = (0, 255, 0) if score <= MOTION_THRESHOLD else (0, 80, 255)
    cv2.rectangle(overlay, (10, 50), (10 + bar_w, 65), bar_col, -1)
    cv2.putText(overlay,
        f"움직임 점수: {score:.1f} (임계값: {MOTION_THRESHOLD})",
        (10, 62), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1)

    return overlay


def print_summary(regular_count: int, motion_count: int):
    """종료 시 저장 결과 요약"""
    files    = sorted(SAVE_DIR.glob("*.png"))
    total_mb = sum(f.stat().st_size for f in files) / (1024 * 1024)

    print("\n" + "=" * 48)
    print(f" 촬영 종료")
    print(f"  정기 촬영  : {regular_count}장")
    print(f"  움직임 감지: {motion_count}장")
    print(f"  총 파일 수 : {len(files)}개  ({total_mb:.1f} MB)")
    print(f"  저장 위치  : {SAVE_DIR.resolve()}")
    print("=" * 48)


print("[OK] 함수 정의 완료")

## 4. 움직임 감지 원리 확인 (선택)
실제 촬영 전에 `absdiff`가 어떻게 동작하는지 시각적으로 확인합니다  
두 이미지의 차이가 클수록 흰색으로 표시됩니다

In [ ]:
import matplotlib.pyplot as plt

# 웹캠에서 0.5초 간격으로 두 장 찍어서 차이 시각화
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("웹캠 없음 — 이 셀은 건너뛰세요")
else:
    ret, frame1 = cap.read()
    time.sleep(0.5)          # 0.5초 후 두 번째 프레임
    ret, frame2 = cap.read()
    cap.release()

    gray1 = cv2.GaussianBlur(
        cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY), (21, 21), 0)
    gray2 = cv2.GaussianBlur(
        cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY), (21, 21), 0)

    diff  = cv2.absdiff(gray1, gray2)
    score = float(np.mean(diff))

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].imshow(cv2.cvtColor(frame1, cv2.COLOR_BGR2RGB))
    axes[0].set_title("프레임 1")
    axes[1].imshow(cv2.cvtColor(frame2, cv2.COLOR_BGR2RGB))
    axes[1].set_title("프레임 2 (0.5초 후)")
    axes[2].imshow(diff, cmap="hot")
    axes[2].set_title(f"차이(absdiff) — 점수: {score:.2f}")
    for ax in axes:
        ax.axis("off")

    plt.suptitle(
        f"움직임 점수 {score:.2f}  |  임계값 {MOTION_THRESHOLD}  →  "
        + ("감지됨 🔴" if score > MOTION_THRESHOLD else "감지 안됨 🟢"),
        fontsize=12
    )
    plt.tight_layout()
    plt.show()

## 5-A. 정기 촬영 + 움직임 감지 실행 — Kinect 모드
종료: 미리보기 창에서 `q` 키

In [ ]:
if not KINECT_AVAILABLE:
    print("Kinect 미연결 → 아래 5-B 셀(웹캠 모드)을 실행하세요")
else:
    pykinect.initialize_libraries()
    device_config = pykinect.default_configuration
    device_config.color_resolution = pykinect.K4A_COLOR_RESOLUTION_1080P
    device_config.depth_mode       = pykinect.K4A_DEPTH_MODE_NFOV_UNBINNED
    device = pykinect.start_device(config=device_config)

    print(f"[Kinect] 연결 성공!")
    print(f"정기 촬영: {INTERVAL_SEC}초마다  |  움직임 임계값: {MOTION_THRESHOLD}")
    print("미리보기 창에서 'q' 키를 누르면 종료됩니다\n")

    prev_gray         = None
    last_capture_time = 0
    regular_count     = 0
    motion_count      = 0
    last_score        = 0.0

    try:
        while True:
            capture                = device.update()
            ret_color, color_image = capture.get_color_image()
            ret_depth, depth_image = capture.get_depth_image()

            if not ret_color or not ret_depth:
                time.sleep(0.1)
                continue

            now       = time.time()
            next_in   = max(0, INTERVAL_SEC - (now - last_capture_time))
            depth_vis = depth_to_colormap(depth_image)
            curr_gray = cv2.GaussianBlur(
                cv2.cvtColor(color_image, cv2.COLOR_BGR2GRAY), (21, 21), 0)
            saved     = False

            # ── 정기 촬영 ──────────────────────────
            if now - last_capture_time >= INTERVAL_SEC:
                path = save_images(color_image, depth_vis, "regular")
                regular_count    += 1
                last_capture_time = now
                saved             = True
                print(f"[{datetime.now().strftime('%H:%M:%S')}] "
                      f"[정기 #{regular_count}] → {path}")

            # ── 움직임 감지 촬영 ───────────────────
            if prev_gray is not None:
                is_motion, score = detect_motion(prev_gray, curr_gray)
                last_score       = score

                if is_motion and not saved and (now - last_motion_time) >= MOTION_COOLDOWN_SEC:
                    path = save_images(color_image, depth_vis, "motion")
                    motion_count += 1
                    print(f"[{datetime.now().strftime('%H:%M:%S')}] "
                          f"[움직임 #{motion_count}] score={score:.1f} → {path}")

            prev_gray = curr_gray

            # ── 미리보기 ───────────────────────────
            if SHOW_PREVIEW:
                preview = draw_overlay(
                    color_image, next_in,
                    regular_count, motion_count, last_score
                )
                cv2.imshow("RGB (q: 종료)",
                           cv2.resize(preview, (1280, 720)))
                cv2.imshow("Depth",
                           cv2.resize(depth_vis, (640, 360)))

            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

            time.sleep(0.5)

    except KeyboardInterrupt:
        print("[종료]")
    finally:
        device.close()
        cv2.destroyAllWindows()
        print_summary(regular_count, motion_count)

## 5-B. 정기 촬영 + 움직임 감지 실행 — 웹캠 모드
종료: 미리보기 창에서 `q` 키

In [ ]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("[오류] 웹캠 연결 실패")
else:
    print(f"[웹캠] 정기 촬영: {INTERVAL_SEC}초마다  |  움직임 임계값: {MOTION_THRESHOLD}")
    print("미리보기 창에서 'q' 키를 누르면 종료됩니다\n")

    prev_gray         = None
    last_capture_time = 0
    regular_count     = 0
    motion_count      = 0
    last_score        = 0.0

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            now       = time.time()
            next_in   = max(0, INTERVAL_SEC - (now - last_capture_time))
            gray      = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            depth_sim = cv2.applyColorMap(gray, cv2.COLORMAP_JET)
            curr_gray = cv2.GaussianBlur(gray, (21, 21), 0)
            saved     = False

            # ── 정기 촬영 ──────────────────────────
            if now - last_capture_time >= INTERVAL_SEC:
                path = save_images(frame, depth_sim, "regular")
                regular_count    += 1
                last_capture_time = now
                saved             = True
                print(f"[{datetime.now().strftime('%H:%M:%S')}] "
                      f"[정기 #{regular_count}] → {path}")

            # ── 움직임 감지 촬영 ───────────────────
            if prev_gray is not None:
                is_motion, score = detect_motion(prev_gray, curr_gray)
                last_score       = score

                if is_motion and not saved and (now - last_motion_time) >= MOTION_COOLDOWN_SEC:
                    path = save_images(frame, depth_sim, "motion")
                    motion_count += 1
                    print(f"[{datetime.now().strftime('%H:%M:%S')}] "
                          f"[움직임 #{motion_count}] score={score:.1f} → {path}")

            prev_gray = curr_gray

            # ── 미리보기 ───────────────────────────
            if SHOW_PREVIEW:
                preview = draw_overlay(
                    frame, next_in,
                    regular_count, motion_count, last_score
                )
                cv2.imshow("웹캠 (q: 종료)", preview)
                cv2.imshow("Depth 시뮬레이션", depth_sim)

            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    except KeyboardInterrupt:
        print("[종료]")
    finally:
        cap.release()
        cv2.destroyAllWindows()
        print_summary(regular_count, motion_count)

## 6. 저장된 이미지 확인
정기 촬영과 움직임 감지 이미지를 구분해서 표시합니다

In [ ]:
import matplotlib.pyplot as plt

regular_files = sorted(SAVE_DIR.glob("*_regular_color.png"))
motion_files  = sorted(SAVE_DIR.glob("*_motion_color.png"))

print(f"정기 촬영 이미지 : {len(regular_files)}장")
print(f"움직임 감지 이미지: {len(motion_files)}장")

def show_images(files, title, max_show=4):
    if not files:
        print(f"{title}: 저장된 이미지 없음")
        return
    show  = files[-max_show:]
    fig, axes = plt.subplots(1, len(show), figsize=(5 * len(show), 3))
    if len(show) == 1:
        axes = [axes]
    for ax, f in zip(axes, show):
        img = cv2.cvtColor(cv2.imread(str(f)), cv2.COLOR_BGR2RGB)
        ax.imshow(cv2.resize(img, (320, 180)))
        ax.set_title(f.name[:18], fontsize=8)
        ax.axis("off")
    plt.suptitle(f"{title} (최근 {len(show)}장)", fontsize=11)
    plt.tight_layout()
    plt.show()

show_images(regular_files, "정기 촬영")
show_images(motion_files,  "움직임 감지")

## 7. 임계값 튜닝 가이드
`MOTION_THRESHOLD` 값을 어떻게 잡을지 판단할 수 있도록  
실제 환경에서의 움직임 점수를 10초간 측정합니다

In [ ]:
import matplotlib.pyplot as plt

print("10초간 움직임 점수를 측정합니다...")
print("측정 중 자연스럽게 움직여 보세요 (뒤척이는 정도)\n")

cap    = cv2.VideoCapture(0)
scores = []
prev_g = None
start  = time.time()

while time.time() - start < 10:
    ret, frame = cap.read()
    if not ret:
        break
    curr_g = cv2.GaussianBlur(
        cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), (21, 21), 0)
    if prev_g is not None:
        score = float(np.mean(cv2.absdiff(prev_g, curr_g)))
        scores.append(score)
    prev_g = curr_g
    time.sleep(0.2)

cap.release()

if scores:
    plt.figure(figsize=(10, 3))
    plt.plot(scores, label="움직임 점수", color="steelblue")
    plt.axhline(MOTION_THRESHOLD, color="red",
                linestyle="--", label=f"현재 임계값 ({MOTION_THRESHOLD})")
    plt.fill_between(range(len(scores)), scores,
                     MOTION_THRESHOLD,
                     where=[s > MOTION_THRESHOLD for s in scores],
                     alpha=0.3, color="red", label="감지 구간")
    plt.xlabel("프레임")
    plt.ylabel("점수")
    plt.title("움직임 점수 측정 결과")
    plt.legend()
    plt.tight_layout()
    plt.show()

    print(f"최솟값: {min(scores):.2f}")
    print(f"최댓값: {max(scores):.2f}")
    print(f"평균값: {np.mean(scores):.2f}")
    print(f"\n권장 임계값: 평균값 + 5~10 사이로 설정하세요")
    print(f"  → 권장: {np.mean(scores) + 7:.0f} ~ {np.mean(scores) + 12:.0f}")